# Image processing and face-authorisation pipeline

This notebook builds the image features and face model, evaluates it, and tests authorized and unauthorized examples.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from image_pipeline import build_image_features_csv, train_face_model, verify_face

## Inspect the image data

Images in `data/images/` are used for training. Filenames determine labels.

In [2]:
images = sorted((ROOT / "data" / "images").iterdir())
[(image.name, image.stat().st_size) for image in images if image.is_file()]

[('flavia.jpeg', 491180),
 ('flavia1.jpeg', 523842),
 ('flavia3.jpeg', 515966),
 ('placide.HEIC', 833778),
 ('placide2.HEIC', 1147337),
 ('placide3.HEIC', 1199985),
 ('Shakira.jpeg', 94154),
 ('shakira2.jpeg', 83781),
 ('shakira3.jpeg', 88594),
 ('unauthorised_smile.png', 661317),
 ('unauthorised_surprised.png', 705502),
 ('unauthorized.jpeg', 81654)]

## Build features and train the model

In [3]:
features_path = build_image_features_csv(
    images_dir=ROOT / "data" / "images",
    output_path=ROOT / "data" / "image_features.csv",
)
metrics = train_face_model(
    features_path=features_path,
    model_path=ROOT / "models" / "face_model.pkl",
    metrics_path=ROOT / "models" / "face_metrics.json",
)
metrics

{'accuracy': 1.0, 'f1_weighted': 1.0, 'log_loss': 0.20260376998286553}

In [4]:
import pandas as pd

features = pd.read_csv(features_path)
features.groupby("label").size().sort_index()

label
Shakira         12
flavia          12
placide         12
unauthorised     8
unauthorized     4
dtype: int64

## Test an authorized example

In [5]:
verify_face(
    ROOT / "data" / "images" / "flavia.jpeg",
    model_path=ROOT / "models" / "face_model.pkl",
)

{'recognized': True, 'userId': 'flavia', 'confidence': 0.93}

## Test unauthorized examples

These images are included as the `unauthorized` class. The application still denies access because that class has no customer profile.

In [6]:
unauthorized_paths = [
    ROOT / "data" / "images" / "unauthorized.jpeg",
    ROOT / "data" / "images" / "unauthorised_smile.png",
    ROOT / "data" / "images" / "unauthorised_surprised.png",
]

{
    path.name: verify_face(path, model_path=ROOT / "models" / "face_model.pkl")
    for path in unauthorized_paths
}

{'unauthorized.jpeg': {'recognized': True,
  'userId': 'unauthorized',
  'confidence': 0.96},
 'unauthorised_smile.png': {'recognized': True,
  'userId': 'unauthorised',
  'confidence': 0.995},
 'unauthorised_surprised.png': {'recognized': True,
  'userId': 'unauthorised',
  'confidence': 0.96}}